In [1]:
import warnings
warnings.filterwarnings("ignore")
import random
import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.cluster import KMeans
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import MinMaxScaler

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D2 in response_OUS
data = list(OUS_D2['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D2, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D2 
clinical_train.isnull().sum().sum()

0

## Test dataset: MAASTRO 

In [6]:
(MAASTRO_D2['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [7]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [8]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D2['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 

In [9]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D2, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]

In [10]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [11]:
# Check if some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,shape_MinorAxisLength,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event


In [12]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [13]:
# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]
lower, upper = np.percentile(y_MAASTRO['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

X_train:  (139, 374)
y_train:  (139,)


(99, 376)

# Feature Selection: RENT

In [14]:
selected_features = ["shape_Sphericity",
                     "glrlm_HighGrayLevelRunEmphasis_PET_c04",
                     "shape_MajorAxisLength",
                     "LBP_102_PET"]

# Selecting features in the DataFrame
X_rent = X[selected_features]

In [15]:
# Selecting features in the DataFrame
X_rent = X[selected_features]
X_new = X_rent.copy()

X_MAASTRO_rent = X_MAASTRO[selected_features]
MAASTRO_new = X_MAASTRO_rent.copy()

# Standardization

In [16]:
# Standardize X_new, the new data with the selected features only 
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = MinMaxScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [17]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [18]:
X_new

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.761164,16.969770,42.073251,0.000000
1,0.697049,15.598394,24.613845,0.000000
2,0.565792,17.334294,48.030294,0.000034
3,0.684364,14.009277,25.589900,0.000000
4,0.503142,21.202180,34.684750,0.000199
...,...,...,...,...
134,0.742102,16.110383,33.069705,0.000000
135,0.722918,21.249575,41.043692,0.000000
136,0.652963,14.873884,36.618802,0.000000
137,0.724255,21.648860,45.870392,0.000000


In [19]:
X_new_std

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.828413,0.487448,0.350904,0.000000
1,0.645006,0.388474,0.123069,0.000000
2,0.269537,0.513756,0.428640,0.066875
3,0.608721,0.273786,0.135806,0.000000
4,0.090323,0.792905,0.254489,0.386334
...,...,...,...,...
134,0.773882,0.425425,0.233413,0.000000
135,0.719008,0.796326,0.337469,0.000000
136,0.518897,0.336186,0.279727,0.000000
137,0.722831,0.825142,0.400455,0.000000


In [20]:
MAASTRO_new

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.668072,19.034156,50.002093,0.000026
1,0.669961,11.392444,41.753334,0.000167
2,0.624081,14.567421,44.375483,0.000057
3,0.577624,13.477331,46.115989,0.000000
4,0.630933,16.365554,54.394967,0.000000
...,...,...,...,...
94,0.671754,17.492295,34.218615,0.000000
95,0.632189,14.267900,51.046869,0.000000
96,0.645548,12.143752,50.417953,0.000000
97,0.727488,15.300229,44.901412,0.000000


In [21]:
MAASTRO_new_std

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.562115,0.636436,0.454371,0.050863
1,0.567520,0.084927,0.346729,0.324583
2,0.436277,0.314068,0.380947,0.110937
3,0.303383,0.235395,0.403660,0.000000
4,0.455879,0.443841,0.511695,0.000000
...,...,...,...,...
94,0.572648,0.525159,0.248406,0.000000
95,0.459472,0.292451,0.468005,0.000000
96,0.497683,0.139149,0.459798,0.000000
97,0.732079,0.366955,0.387810,0.000000


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [22]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-15 14:17:34,098] A new study created in memory with name: no-name-3175c5c7-c5a5-40ed-a2b0-bebc277cccce


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6796536796536796
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.7341772151898734


[I 2024-04-15 14:17:50,421] A new study created in memory with name: no-name-aaa320ef-24a0-4ff5-9286-87c68c1fd205


Fold 5 C-index: 0.6948356807511737
[I 2024-04-15 14:17:50,402] Trial 0 finished with value: 0.7358264523738474 and parameters: {}. Best is trial 0 with value: 0.7358264523738474.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7358264523738474], datetime_start=datetime.datetime(2024, 4, 15, 14, 17, 34, 564424), datetime_complete=datetime.datetime(2024, 4, 15, 14, 17, 50, 401184), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7358264523738474


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.17932951816141288
Fold 2 IBS: 0.15173912855888322
Fold 3 IBS: 0.17662964771229905
Fold 4 IBS: 0.15303709600503954
Fold 5 IBS: 0.19681553106978886
[I 2024-04-15 14:17:51,576] Trial 0 finished with value: 0.17151018430148474 and parameters: {}. Best is trial 0 with value: 0.17151018430148474.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.17151018430148474], datetime_start=datetime.datetime(2024, 4, 15, 14, 17, 50, 456858), datetime_complete=datetime.datetime(2024, 4, 15, 14, 17, 51, 575091), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.17151018430148474


In [23]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [24]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.736
train_ibs:  0.172


#### Test

In [25]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [26]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO,y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.567
IBS score: 0.259


In [27]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [28]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge 

#### Train

In [29]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 14:17:52,217] A new study created in memory with name: no-name-73cb4d2d-81c5-45ac-9ff5-27d5aa11b00b


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6796536796536796
Fold 2 C-index: 0.7254464285714286
Fold 3 C-index: 0.75
Fold 4 C-index: 0.7130801687763713


[I 2024-04-15 14:17:52,654] A new study created in memory with name: no-name-81626e37-6f22-4534-be33-75166cebf5d3


Fold 5 C-index: 0.7394366197183099
[I 2024-04-15 14:17:52,645] Trial 0 finished with value: 0.7215233793439578 and parameters: {}. Best is trial 0 with value: 0.7215233793439578.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7215233793439578], datetime_start=datetime.datetime(2024, 4, 15, 14, 17, 52, 310531), datetime_complete=datetime.datetime(2024, 4, 15, 14, 17, 52, 642988), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7215233793439578


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397652066529
Fold 2 IBS: 0.22157791508266436
Fold 3 IBS: 0.20453594548791937
Fold 4 IBS: 0.2247380394966658
Fold 5 IBS: 0.21812431321836134
[I 2024-04-15 14:17:53,153] Trial 0 finished with value: 0.21659054679018022 and parameters: {}. Best is trial 0 with value: 0.21659054679018022.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21659054679018022], datetime_start=datetime.datetime(2024, 4, 15, 14, 17, 52, 713904), datetime_complete=datetime.datetime(2024, 4, 15, 14, 17, 53, 153135), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21659054679018022


In [30]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [31]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.722
train_ibs:  0.217


#### Test

In [32]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [33]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO,y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.565


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [34]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [35]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 14:17:53,580] A new study created in memory with name: no-name-0672198d-bb1d-4887-9ff6-d81adbb8b5c8


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7843137254901961


[I 2024-04-15 14:17:54,827] A new study created in memory with name: no-name-e495d8c5-6698-42d4-985b-0203c88fd869


Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-15 14:17:54,785] Trial 0 finished with value: 0.7357337800920462 and parameters: {}. Best is trial 0 with value: 0.7357337800920462.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7357337800920462], datetime_start=datetime.datetime(2024, 4, 15, 14, 17, 53, 842285), datetime_complete=datetime.datetime(2024, 4, 15, 14, 17, 54, 785355), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7357337800920462


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.17978486969064494
Fold 2 IBS: 0.15091426109195713
Fold 3 IBS: 0.17579948353248076
Fold 4 IBS: 0.15359370021287927
Fold 5 IBS: 0.19563635363486537
[I 2024-04-15 14:17:55,858] Trial 0 finished with value: 0.17114573363256547 and parameters: {}. Best is trial 0 with value: 0.17114573363256547.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.17114573363256547], datetime_start=datetime.datetime(2024, 4, 15, 14, 17, 54, 931933), datetime_complete=datetime.datetime(2024, 4, 15, 14, 17, 55, 855428), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.17114573363256547


In [36]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [37]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.736
train_ibs:  0.171


#### Test 

In [38]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [39]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO,y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.566


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.257


In [40]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [41]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 14:17:57,077] A new study created in memory with name: no-name-5e28e438-45d3-494a-ba9b-5771ac6c7556


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-15 14:17:58,885] Trial 0 finished with value: 0.7367141722489089 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7367141722489089.
Fold 1 C-index: 0.683982683982684
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-15 14:18:00,526] Trial 1 finished with value: 0.7358483713831081 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.7367141722489089.
Fold 1 C-index: 0.683982683982684
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-15 14:18:02,152] Trial 2 finished with value: 0.7358483713831081 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.7367141

Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-15 14:18:28,085] Trial 24 finished with value: 0.7358483713831081 and parameters: {'l1_ratio': 0.4980983035246007}. Best is trial 20 with value: 0.7376070293917663.
Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-15 14:18:28,957] Trial 25 finished with value: 0.7375799731147099 and parameters: {'l1_ratio': 0.7784745121749661}. Best is trial 20 with value: 0.7376070293917663.
Fold 1 C-index: 0.683982683982684
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-15 14:18:29,839] Trial 26 finished with value: 0.7358483713831081 and parameters: {'l1_ratio': 0.3312072877416866}. Best is trial 20 with value: 0.7376070293917663.
Fold 1 C-index: 0.683982683982684
Fold 2 C-

Fold 5 C-index: 0.6948356807511737
[I 2024-04-15 14:18:53,882] Trial 48 finished with value: 0.7367141722489089 and parameters: {'l1_ratio': 0.7406555774839976}. Best is trial 20 with value: 0.7376070293917663.
Fold 1 C-index: 0.683982683982684
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-15 14:18:55,206] Trial 49 finished with value: 0.7358483713831081 and parameters: {'l1_ratio': 0.37275461898476625}. Best is trial 20 with value: 0.7376070293917663.
Fold 1 C-index: 0.683982683982684
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-15 14:18:56,438] Trial 50 finished with value: 0.7358483713831081 and parameters: {'l1_ratio': 0.2311229033161103}. Best is trial 20 with value: 0.7376070293917663.
Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.17939857143571256
Fold 2 IBS: 0.15109163708960624
Fold 3 IBS: 0.1763396348009305
Fold 4 IBS: 0.1537043254337406
Fold 5 IBS: 0.1956773341873156
[I 2024-04-15 14:19:41,053] Trial 0 finished with value: 0.17124230058946108 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.17124230058946108.
Fold 1 IBS: 0.1789996220483504
Fold 2 IBS: 0.1512509959799121
Fold 3 IBS: 0.17676137368432185
Fold 4 IBS: 0.15380226909628456
Fold 5 IBS: 0.1958162328928413
[I 2024-04-15 14:19:41,966] Trial 1 finished with value: 0.17132609874034205 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.17124230058946108.
Fold 1 IBS: 0.17896259000470527
Fold 2 IBS: 0.15127585271929875
Fold 3 IBS: 0.17680557714583164
Fold 4 IBS: 0.1538303954940592
Fold 5 IBS: 0.19574833931247274
[I 2024-04-15 14:19:42,993] Trial 2 finished with value: 0.17132455093527352 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.1712423005894610

Fold 1 IBS: 0.17965742075121308
Fold 2 IBS: 0.15099032513038543
Fold 3 IBS: 0.17599680129377795
Fold 4 IBS: 0.15366012000797843
Fold 5 IBS: 0.1956677392109295
[I 2024-04-15 14:19:57,989] Trial 25 finished with value: 0.1711944812788569 and parameters: {'l1_ratio': 0.901420099010056}. Best is trial 11 with value: 0.17114513223300334.
Fold 1 IBS: 0.1794389650554732
Fold 2 IBS: 0.15106763780110619
Fold 3 IBS: 0.1762793396084056
Fold 4 IBS: 0.15373105430206635
Fold 5 IBS: 0.19566872499552762
[I 2024-04-15 14:19:58,615] Trial 26 finished with value: 0.1712371443525158 and parameters: {'l1_ratio': 0.7417169602645222}. Best is trial 11 with value: 0.17114513223300334.
Fold 1 IBS: 0.1793411272124821
Fold 2 IBS: 0.1511109403370602
Fold 3 IBS: 0.1763970994799986
Fold 4 IBS: 0.15372820534565287
Fold 5 IBS: 0.19567773958708665
[I 2024-04-15 14:19:59,245] Trial 27 finished with value: 0.1712510223924561 and parameters: {'l1_ratio': 0.6494728799686162}. Best is trial 11 with value: 0.171145132233003

Fold 1 IBS: 0.1791599317538455
Fold 2 IBS: 0.151207053531672
Fold 3 IBS: 0.1765943472302714
Fold 4 IBS: 0.15377042990701825
Fold 5 IBS: 0.1957469037582549
[I 2024-04-15 14:20:13,943] Trial 50 finished with value: 0.17129573323621242 and parameters: {'l1_ratio': 0.47241766561900594}. Best is trial 11 with value: 0.17114513223300334.
Fold 1 IBS: 0.17973942654319142
Fold 2 IBS: 0.15095358038819007
Fold 3 IBS: 0.1759032111479001
Fold 4 IBS: 0.15363386793868994
Fold 5 IBS: 0.1956511902619194
[I 2024-04-15 14:20:14,594] Trial 51 finished with value: 0.17117625525597818 and parameters: {'l1_ratio': 0.9495136153828908}. Best is trial 11 with value: 0.17114513223300334.
Fold 1 IBS: 0.17977528625638908
Fold 2 IBS: 0.15097771852124492
Fold 3 IBS: 0.17580690208447577
Fold 4 IBS: 0.1536041711478401
Fold 5 IBS: 0.19572183222058723
[I 2024-04-15 14:20:15,159] Trial 52 finished with value: 0.17117718204610743 and parameters: {'l1_ratio': 0.9890190374631385}. Best is trial 11 with value: 0.171145132233

Fold 1 IBS: 0.17964920996955602
Fold 2 IBS: 0.15098471273560565
Fold 3 IBS: 0.17600281382450383
Fold 4 IBS: 0.15363795085093587
Fold 5 IBS: 0.1956511950989893
[I 2024-04-15 14:20:33,888] Trial 75 finished with value: 0.17118517649591816 and parameters: {'l1_ratio': 0.8926447980757628}. Best is trial 71 with value: 0.17114478371582048.
Fold 1 IBS: 0.17975440942456136
Fold 2 IBS: 0.15096363948106567
Fold 3 IBS: 0.17589251244197426
Fold 4 IBS: 0.1536086440474439
Fold 5 IBS: 0.19568077974829168
[I 2024-04-15 14:20:34,587] Trial 76 finished with value: 0.17117999702866737 and parameters: {'l1_ratio': 0.9657119709985041}. Best is trial 71 with value: 0.17114478371582048.
Fold 1 IBS: 0.17958298235895545
Fold 2 IBS: 0.15102311499889745
Fold 3 IBS: 0.17608285904146911
Fold 4 IBS: 0.15365396277963647
Fold 5 IBS: 0.19568126430542182
[I 2024-04-15 14:20:35,258] Trial 77 finished with value: 0.17120483669687606 and parameters: {'l1_ratio': 0.8531345030167654}. Best is trial 71 with value: 0.1711447

In [42]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [43]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.738
train_ibs:  0.171


#### Test

In [44]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [45]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO,y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.6127580877814559)

test_cindex : 0.567


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.9956254432215501)

test_ibs:  0.257


In [46]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [47]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-15 14:20:53,002] A new study created in memory with name: no-name-919fe68e-b370-4610-af64-62e471857823


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7467532467532467
Fold 2 C-index: 0.8125
Fold 3 C-index: 0.7769607843137255
Fold 4 C-index: 0.6814345991561181
Fold 5 C-index: 0.7699530516431925
[I 2024-04-15 14:20:59,625] Trial 0 finished with value: 0.7575203363732566 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7575203363732566.
Fold 1 C-index: 0.7445887445887446
Fold 2 C-index: 0.8169642857142857
Fold 3 C-index: 0.75
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.7699530516431925
[I 2024-04-15 14:21:04,738] Trial 1 finished with value: 0.750478431579118 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'max_features': 'sqrt', 'mi

Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.7991071428571429
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.7341772151898734
Fold 5 C-index: 0.7746478873239436
[I 2024-04-15 14:22:11,362] Trial 16 finished with value: 0.765813594477808 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 15, 'min_samples_leaf': 16, 'max_depth': 20, 'n_estimators': 5, 'oob_score': True, 'max_samples': 0.8484814588885312, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.08082385841318356, 'warm_start': True}. Best is trial 14 with value: 0.7907144637948861.
Fold 1 C-index: 0.7597402597402597
Fold 2 C-index: 0.8526785714285714
Fold 3 C-index: 0.7867647058823529
Fold 4 C-index: 0.7236286919831224
Fold 5 C-index: 0.8028169014084507
[I 2024-04-15 14:22:12,408] Trial 17 finished with value: 0.7851258260885514 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 15, 'min_samples_leaf': 15, 'max_depth': 17, 'n_estimators': 106, 'oob_score': True, 'max_samples': 0.98170723046990

Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.8303571428571429
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.7658227848101266
Fold 5 C-index: 0.8356807511737089
[I 2024-04-15 14:22:28,974] Trial 31 finished with value: 0.7982641653072841 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 14, 'min_samples_leaf': 7, 'max_depth': 8, 'n_estimators': 75, 'oob_score': True, 'max_samples': 0.745146001885883, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.05174429879930956, 'warm_start': True}. Best is trial 31 with value: 0.7982641653072841.
Fold 1 C-index: 0.7575757575757576
Fold 2 C-index: 0.8348214285714286
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.7742616033755274
Fold 5 C-index: 0.8309859154929577
[I 2024-04-15 14:22:29,872] Trial 32 finished with value: 0.8032544311992128 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 14, 'min_samples_leaf': 7, 'max_depth': 9, 'n_estimators': 78, 'oob_score': True, 'max_samples': 0.7397628776532771, 'm

...

Fold 5 IBS: 0.17698987901902175
[I 2024-04-15 14:31:06,830] Trial 31 finished with value: 0.1756223060197624 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 17, 'min_samples_leaf': 13, 'max_depth': 1, 'n_estimators': 495, 'oob_score': False, 'max_samples': 0.6141886010044044, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.050889504115644586}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17921499719523448
Fold 2 IBS: 0.14457659998302025
Fold 3 IBS: 0.18892068480933402
Fold 4 IBS: 0.1881785321534129
Fold 5 IBS: 0.17366220512930025
[I 2024-04-15 14:31:23,107] Trial 32 finished with value: 0.17491060385406038 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 19, 'min_samples_leaf': 11, 'max_depth': 2, 'n_estimators': 473, 'oob_score': False, 'max_samples': 0.7145937356077748, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.040680386261001684}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17839650482493216
Fold 2 IBS: 0.

Fold 1 IBS: 0.16980857091041326
Fold 2 IBS: 0.15719115142034723
Fold 3 IBS: 0.1767449129839921
Fold 4 IBS: 0.19510331227382113
Fold 5 IBS: 0.1831755952242518
[I 2024-04-15 14:33:46,859] Trial 47 finished with value: 0.17640470856256513 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 20, 'min_samples_leaf': 11, 'max_depth': 2, 'n_estimators': 479, 'oob_score': False, 'max_samples': 0.8914383434724882, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.24824101632322468}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.19349637716238946
Fold 2 IBS: 0.18021514079410297
Fold 3 IBS: 0.1797766992061227
Fold 4 IBS: 0.20805913242700583
Fold 5 IBS: 0.19958771263889286
[I 2024-04-15 14:33:59,129] Trial 48 finished with value: 0.19222701244570276 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 14, 'min_samples_leaf': 6, 'max_depth': 3, 'n_estimators': 439, 'oob_score': False, 'max_samples': 0.9967788477130068, 'max_features': 'auto', 'min_weight_fraction_

Fold 5 IBS: 0.17601995444169838
[I 2024-04-15 14:36:15,907] Trial 62 finished with value: 0.1766496500814655 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 16, 'min_samples_leaf': 9, 'max_depth': 3, 'n_estimators': 298, 'oob_score': True, 'max_samples': 0.6969767031486694, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.11947149985769817}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17325965795999854
Fold 2 IBS: 0.16035062956465257
Fold 3 IBS: 0.17539882197990753
Fold 4 IBS: 0.193362443315604
Fold 5 IBS: 0.1796554283287958
[I 2024-04-15 14:36:23,106] Trial 63 finished with value: 0.17640539622979168 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 15, 'min_samples_leaf': 10, 'max_depth': 1, 'n_estimators': 311, 'oob_score': True, 'max_samples': 0.6304198495967799, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.16702154024396243}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17417602273955624
Fold 2 IBS: 0.1480464

Fold 1 IBS: 0.16818314645426713
Fold 2 IBS: 0.15369061634554518
Fold 3 IBS: 0.18213459050433092
Fold 4 IBS: 0.19123211966917278
Fold 5 IBS: 0.17979973767912885
[I 2024-04-15 14:38:20,595] Trial 78 finished with value: 0.17500804213048896 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 11, 'max_depth': 2, 'n_estimators': 487, 'oob_score': False, 'max_samples': 0.4897248144615137, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.053996872945124036}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.1682237549262069
Fold 2 IBS: 0.15295334400832536
Fold 3 IBS: 0.18214608764478374
Fold 4 IBS: 0.19238806563941324
Fold 5 IBS: 0.18028732465861996
[I 2024-04-15 14:38:30,295] Trial 79 finished with value: 0.17519971537546983 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 11, 'max_depth': 11, 'n_estimators': 500, 'oob_score': False, 'max_samples': 0.49575398669591153, 'max_features': 'auto', 'min_weight_frac

Fold 5 IBS: 0.1747309238915691
[I 2024-04-15 14:40:41,282] Trial 93 finished with value: 0.1752048898255005 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 19, 'min_samples_leaf': 10, 'max_depth': 4, 'n_estimators': 437, 'oob_score': False, 'max_samples': 0.5630698614554802, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.0944985972942633}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17026943747099219
Fold 2 IBS: 0.15255352423829832
Fold 3 IBS: 0.18160069813968083
Fold 4 IBS: 0.19096533404342483
Fold 5 IBS: 0.1822030877543707
[I 2024-04-15 14:40:50,088] Trial 94 finished with value: 0.17551841632935336 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 18, 'min_samples_leaf': 11, 'max_depth': 2, 'n_estimators': 410, 'oob_score': False, 'max_samples': 0.5751544947591161, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.14409687866880386}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17475332961241374
Fold 2 IBS: 0.1452

In [48]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [49]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.846
train_ibs:  0.171


#### Test

In [50]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [51]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO,y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=13, max_features=None, max_leaf_nodes=12,
                     max_samples=0.48042970450395145, min_samples_leaf=1,
                     min_samples_split=4,
                     min_weight_fraction_leaf=0.0025914663927405594,
                     n_estimators=289, random_state=123, warm_start=True)

test_cindex:  0.602


RandomSurvivalForest(max_depth=1, max_features='auto', max_leaf_nodes=18,
                     max_samples=0.7351810575897255, min_samples_leaf=11,
                     min_samples_split=2,
                     min_weight_fraction_leaf=0.008231293935378081,
                     n_estimators=2, random_state=123)

test_ibs:  0.233


In [52]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [53]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 14:41:32,990] A new study created in memory with name: no-name-60fe6f95-0441-431a-bf7e-5e30f72567f1


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.70995670995671
Fold 2 C-index: 0.8348214285714286
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.7257383966244726
Fold 5 C-index: 0.755868544600939
[I 2024-04-15 14:41:34,450] Trial 0 finished with value: 0.763120153205612 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.763120153205612.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 14:41:38,828] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}. Be

Fold 1 C-index: 0.6645021645021645
Fold 2 C-index: 0.8191964285714286
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.7088607594936709
Fold 5 C-index: 0.676056338028169
[I 2024-04-15 14:42:22,394] Trial 16 finished with value: 0.7335270596877141 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is trial 12 with value: 0.7656344262692516.
Fold 1 C-index: 0.7445887445887446
Fold 2 C-index: 0.8348214285714286
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.7046413502109705
Fold 5 C-index: 0.7323943661971831
[I 2024-04-15 14:42:23,686] Trial 17 finished with value: 0.7591715308548418 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 12, 'n_estimators': 318, 'oob_score': False, 'warm_start': True, 'max_featu

Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.84375
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.7046413502109705
Fold 5 C-index: 0.7276995305164319
[I 2024-04-15 14:42:57,095] Trial 31 finished with value: 0.7601328692954678 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 16, 'min_samples_leaf': 15, 'max_depth': 3, 'n_estimators': 375, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.9462932212483206, 'min_weight_fraction_leaf': 0.12021009726526694}. Best is trial 19 with value: 0.7663281634231028.
Fold 1 C-index: 0.7272727272727273
Fold 2 C-index: 0.84375
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.7215189873417721
Fold 5 C-index: 0.7323943661971831
[I 2024-04-15 14:42:59,357] Trial 32 finished with value: 0.760869569103513 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 17, 'min_samples_leaf': 14, 'max_depth': 3, 'n_estimators': 462, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_sa

...

Fold 1 IBS: 0.18923073618097216
Fold 2 IBS: 0.17632006586851645
Fold 3 IBS: 0.18707067058836963
Fold 4 IBS: 0.19307240934736852
Fold 5 IBS: 0.19343354994654252
[I 2024-04-15 14:48:19,415] Trial 30 finished with value: 0.18782548638635385 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 355, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7262614295600427, 'min_weight_fraction_leaf': 0.020774312173092817}. Best is trial 23 with value: 0.18218668005091396.
Fold 1 IBS: 0.1949036294792145
Fold 2 IBS: 0.1886371865011974
Fold 3 IBS: 0.19117863815442443
Fold 4 IBS: 0.2001198684681372
Fold 5 IBS: 0.19929856633981852
[I 2024-04-15 14:48:25,248] Trial 31 finished with value: 0.19482757778855841 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 2, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 396, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.

Fold 1 IBS: 0.18107213977138975
Fold 2 IBS: 0.17375539456626857
Fold 3 IBS: 0.18442566650093445
Fold 4 IBS: 0.19068424307356363
Fold 5 IBS: 0.18831741970696433
[I 2024-04-15 14:50:26,643] Trial 45 finished with value: 0.18365097272382416 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 11, 'min_samples_leaf': 2, 'max_depth': 17, 'n_estimators': 366, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.9992649320581628, 'min_weight_fraction_leaf': 0.15624439720920977}. Best is trial 44 with value: 0.17800229474539986.
Fold 1 IBS: 0.19953982633918327
Fold 2 IBS: 0.19967504623742535
Fold 3 IBS: 0.19245042867286613
Fold 4 IBS: 0.2085347403492104
Fold 5 IBS: 0.20542311562702958
[I 2024-04-15 14:50:36,570] Trial 46 finished with value: 0.20112463144514292 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 10, 'min_samples_leaf': 4, 'max_depth': 15, 'n_estimators': 471, 'oob_score': True, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.8

Fold 1 IBS: 0.17659457791938485
Fold 2 IBS: 0.15948702371575116
Fold 3 IBS: 0.18725320428102982
Fold 4 IBS: 0.18157027728642194
Fold 5 IBS: 0.18138733472020935
[I 2024-04-15 14:52:12,786] Trial 60 finished with value: 0.17725848358455942 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 5, 'min_samples_leaf': 6, 'max_depth': 15, 'n_estimators': 346, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.7897702249078459, 'min_weight_fraction_leaf': 0.017004106952235656}. Best is trial 57 with value: 0.1764459724345519.
Fold 1 IBS: 0.17815851130122365
Fold 2 IBS: 0.15937759601308119
Fold 3 IBS: 0.18671233531081954
Fold 4 IBS: 0.18030514263303932
Fold 5 IBS: 0.18041925981213724
[I 2024-04-15 14:52:20,177] Trial 61 finished with value: 0.1769945690140602 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 5, 'min_samples_leaf': 6, 'max_depth': 15, 'n_estimators': 340, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.861

Fold 1 IBS: 0.21400073415496554
Fold 2 IBS: 0.2208296768873117
Fold 3 IBS: 0.20504439468604338
Fold 4 IBS: 0.22482948519570425
Fold 5 IBS: 0.21793226667188703
[I 2024-04-15 14:53:44,420] Trial 75 finished with value: 0.21652731151918242 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 6, 'min_samples_leaf': 7, 'max_depth': 11, 'n_estimators': 301, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.28939609623192764, 'min_weight_fraction_leaf': 0.38792353869222}. Best is trial 73 with value: 0.17587600438953827.
Fold 1 IBS: 0.18092605908791307
Fold 2 IBS: 0.1626714178491272
Fold 3 IBS: 0.1886398695722679
Fold 4 IBS: 0.18093264680250482
Fold 5 IBS: 0.18045471816506836
[I 2024-04-15 14:53:48,837] Trial 76 finished with value: 0.17872494229537628 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 4, 'min_samples_leaf': 5, 'max_depth': 14, 'n_estimators': 229, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.82756

Fold 1 IBS: 0.18417873567489768
Fold 2 IBS: 0.17215507074054423
Fold 3 IBS: 0.18759974758071274
Fold 4 IBS: 0.1889870504957803
Fold 5 IBS: 0.18950903071510605
[I 2024-04-15 14:55:08,534] Trial 90 finished with value: 0.1844859270414082 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 4, 'min_samples_leaf': 4, 'max_depth': 10, 'n_estimators': 262, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.9722689668575789, 'min_weight_fraction_leaf': 0.013698489831834436}. Best is trial 73 with value: 0.17587600438953827.
Fold 1 IBS: 0.17828049657703648
Fold 2 IBS: 0.15670442146325114
Fold 3 IBS: 0.1885595963781235
Fold 4 IBS: 0.17837564239035944
Fold 5 IBS: 0.17924664597888262
[I 2024-04-15 14:55:14,681] Trial 91 finished with value: 0.17623336055753064 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 5, 'max_depth': 15, 'n_estimators': 308, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.83

In [54]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [55]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.791
train_ibs:  0.176


#### Test

In [56]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [57]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO,y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=19, max_features=1, max_leaf_nodes=12,
                   max_samples=0.21404335777324493, min_samples_leaf=1,
                   min_samples_split=4,
                   min_weight_fraction_leaf=0.0013133682808468272,
                   n_estimators=405, random_state=123, warm_start=True)

C-index score: 0.572


ExtraSurvivalTrees(max_depth=14, max_features=None, max_leaf_nodes=6,
                   max_samples=0.8396800309305413, min_samples_leaf=5,
                   min_samples_split=5,
                   min_weight_fraction_leaf=0.02494499808293557,
                   n_estimators=305, oob_score=True, random_state=123)

IBS: 0.221


In [58]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [59]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-15 14:56:03,866] A new study created in memory with name: no-name-716f8b5d-efe7-4ee0-bfd9-28c07b2c527e


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 14:56:34,073] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 14:56:49,999] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 15:05:37,954] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7610259439316011.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 15:06:46,698] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 15:24:29,806] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8840318412875596, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.27382248438555523, 'n_estimators': 385, 'criterion': 'squared_error', 'ccp_alpha': 2.0183033060186855, 'min_weight_fraction_leaf': 0.33643534713187806, 'max_features': 'auto', 'min_impurity_decrease': 5.889654690360788e-06, 'validation_fraction': 0.8062622793646869, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 9 with value: 0.7610259439316011.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 15:26:00,845] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7565765917190008, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.4300954216773497, 'n_estimators': 440, 'criterion': 'friedman

...

Fold 3 IBS: 0.20286393718880646
Fold 4 IBS: 0.22391249179729633
Fold 5 IBS: 0.21519882725417372
[I 2024-04-15 17:28:47,183] Trial 55 finished with value: 0.21426660724128474 and parameters: {'subsample': 0.9703353679292269, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.2676210325613897, 'n_estimators': 481, 'criterion': 'squared_error', 'ccp_alpha': 0.036860238643527846, 'min_weight_fraction_leaf': 0.21798842867076448, 'max_features': None, 'min_impurity_decrease': 4.086647023052284e-07, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 18, 'max_leaf_nodes': 15, 'min_samples_leaf': 10, 'max_depth': 4}. Best is trial 42 with value: 0.21286195228463267.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-15 17:29:28,697] Trial 56 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.9556274374724505, 'learning_rate': 0.022777236

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-15 17:31:59,946] Trial 66 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9019393211486977, 'learning_rate': 0.09984676062907398, 'dropout_rate': 0.14697195606329974, 'n_estimators': 8, 'criterion': 'squared_error', 'ccp_alpha': 0.7042671412306138, 'min_weight_fraction_leaf': 0.07018529613097008, 'max_features': None, 'min_impurity_decrease': 3.137142110783047e-07, 'validation_fraction': 0.779814650235569, 'min_samples_split': 18, 'max_leaf_nodes': 13, 'min_samples_leaf': 8, 'max_depth': 5}. Best is trial 57 with value: 0.21250989041794505.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-15 17:32:04,341] Trial 67 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.922035492452401, 'learning_rate': 0.021388128948998

Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609557
[I 2024-04-15 17:33:20,543] Trial 77 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.834194505922225, 'learning_rate': 0.023289065494024846, 'dropout_rate': 0.5946683576609367, 'n_estimators': 105, 'criterion': 'friedman_mse', 'ccp_alpha': 5.8752783688356525, 'min_weight_fraction_leaf': 0.10946812472698092, 'max_features': 'log2', 'min_impurity_decrease': 3.863869195457471e-05, 'validation_fraction': 0.8140600247331974, 'min_samples_split': 16, 'max_leaf_nodes': 17, 'min_samples_leaf': 12, 'max_depth': 1}. Best is trial 70 with value: 0.21173544073366624.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.2181243152560957
[I 2024-04-15 17:33:22,915] Trial 78 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9202091045129451, 'learning_rate': 0.019797687069

Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609582
[I 2024-04-15 17:34:04,381] Trial 88 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.45521815335975935, 'learning_rate': 0.024635453467641497, 'dropout_rate': 0.22592407076200724, 'n_estimators': 225, 'criterion': 'friedman_mse', 'ccp_alpha': 0.5932990711529844, 'min_weight_fraction_leaf': 0.05795880477433249, 'max_features': 0.1, 'min_impurity_decrease': 0.00010910005120952281, 'validation_fraction': 0.86583149343058, 'min_samples_split': 13, 'max_leaf_nodes': 17, 'min_samples_leaf': 6, 'max_depth': 6}. Best is trial 82 with value: 0.2114835599960237.
Fold 1 IBS: 0.20406423720489414
Fold 2 IBS: 0.20607296701907554
Fold 3 IBS: 0.19811121797640907
Fold 4 IBS: 0.22011942798029224
Fold 5 IBS: 0.20776003972546755
[I 2024-04-15 17:34:07,357] Trial 89 finished with value: 0.2072255779812277 and parameters: {'subsample': 0.9081334064688743, 'learning_rate': 0.032246585462716706, 'dropout_rate': 0.10084377



* Best trial for IBS: 
 FrozenTrial(number=89, state=TrialState.COMPLETE, values=[0.2072255779812277], datetime_start=datetime.datetime(2024, 4, 15, 17, 34, 4, 389376), datetime_complete=datetime.datetime(2024, 4, 15, 17, 34, 7, 356130), params={'subsample': 0.9081334064688743, 'learning_rate': 0.032246585462716706, 'dropout_rate': 0.10084377376256726, 'n_estimators': 92, 'criterion': 'friedman_mse', 'ccp_alpha': 0.019065157478907357, 'min_weight_fraction_leaf': 0.21843803937299006, 'max_features': None, 'min_impurity_decrease': 6.552978044824526e-05, 'validation_fraction': 0.8292287698601738, 'min_samples_split': 18, 'max_leaf_nodes': 15, 'min_samples_leaf': 9, 'max_depth': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'subsample': FloatDistribution(high=1.0, log=False, low=0.1, step=None), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.001, step=None), 'dropout_rate': FloatDistribution(high=1.0, log=False, low=0.1, step=None), 'n_estimato

In [60]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [61]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.765
train_ibs:  0.207


#### Test

In [62]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [63]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO,y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.046986154574499193,
                                 dropout_rate=0.9141084507237711,
                                 learning_rate=0.005228819896792163,
                                 max_leaf_nodes=13,
                                 min_impurity_decrease=0.00017501601948817654,
                                 min_samples_leaf=15, min_samples_split=11,
                                 min_weight_fraction_leaf=0.2724947676380759,
                                 n_estimators=395, random_state=123,
                                 subsample=0.6128161811459047,
                                 validation_fraction=0.542037097833947)

C-index score: 0.579


GradientBoostingSurvivalAnalysis(ccp_alpha=0.019065157478907357,
                                 dropout_rate=0.10084377376256726,
                                 learning_rate=0.032246585462716706,
                                 max_leaf_nodes=15,
                                 min_impurity_decrease=6.552978044824526e-05,
                                 min_samples_leaf=9, min_samples_split=18,
                                 min_weight_fraction_leaf=0.21843803937299006,
                                 n_estimators=92, random_state=123,
                                 subsample=0.9081334064688743,
                                 validation_fraction=0.8292287698601738)

IBS: 0.218


In [64]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [65]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 17:34:46,035] A new study created in memory with name: no-name-2f0b01e5-4145-4fc5-9240-74e2fe23bdd2


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.696078431372549
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6713615023474179
[I 2024-04-15 17:34:46,748] Trial 0 finished with value: 0.6779611770557914 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6779611770557914.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7107843137254902
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6713615023474179
[I 2024-04-15 17:34:53,580] Trial 1 finished with value: 0.6809023535263796 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 1 with value: 0.6809023535263796.
Fold 1 C-index: 0.5584415584415584
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.6470588235294118
Fold 4

Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7058823529411765
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6713615023474179
[I 2024-04-15 17:35:50,576] Trial 19 finished with value: 0.6799219613695169 and parameters: {'subsample': 0.8350722272141606, 'dropout_rate': 0.99497601792777, 'n_estimators': 431, 'learning_rate': 0.08172742798527305}. Best is trial 10 with value: 0.6818827456832424.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6713615023474179
[I 2024-04-15 17:35:52,831] Trial 20 finished with value: 0.6818827456832424 and parameters: {'subsample': 0.9139336698304192, 'dropout_rate': 0.6483171405010271, 'n_estimators': 296, 'learning_rate': 0.038682181981402025}. Best is trial 10 with value: 0.6818827456832424.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7156862745098039
Fo

Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6713615023474179
[I 2024-04-15 17:36:52,479] Trial 38 finished with value: 0.6818827456832424 and parameters: {'subsample': 0.9485832819053552, 'dropout_rate': 0.13272164755980653, 'n_estimators': 337, 'learning_rate': 0.09007603128714806}. Best is trial 10 with value: 0.6818827456832424.
Fold 1 C-index: 0.5887445887445888
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.6225490196078431
Fold 4 C-index: 0.7130801687763713
Fold 5 C-index: 0.6948356807511737
[I 2024-04-15 17:36:53,327] Trial 39 finished with value: 0.6729490344331384 and parameters: {'subsample': 0.22212752554206305, 'dropout_rate': 0.6881141432789704, 'n_estimators': 145, 'learning_rate': 0.08562293525961606}. Best is trial 10 with value: 0.6818827456832424.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7156862745098039

...

Fold 2 IBS: 0.17892642660722471
Fold 3 IBS: 0.18267258430610997
Fold 4 IBS: 0.1910460474072693
Fold 5 IBS: 0.18440907707340753
[I 2024-04-15 17:41:44,938] Trial 19 finished with value: 0.18855038194134885 and parameters: {'subsample': 0.8447766709063648, 'dropout_rate': 0.12690048952433258, 'n_estimators': 281, 'learning_rate': 0.019107774444910287}. Best is trial 17 with value: 0.1781303871639809.
Fold 1 IBS: 0.21556848748356808
Fold 2 IBS: 0.15529895506013994
Fold 3 IBS: 0.17400594079322942
Fold 4 IBS: 0.17306227825948983
Fold 5 IBS: 0.1731153967884686
[I 2024-04-15 17:41:47,640] Trial 20 finished with value: 0.17821021167697917 and parameters: {'subsample': 0.8714518330888589, 'dropout_rate': 0.24699216442522764, 'n_estimators': 245, 'learning_rate': 0.04061880568044879}. Best is trial 17 with value: 0.1781303871639809.
Fold 1 IBS: 0.21383415966041183
Fold 2 IBS: 0.15768556648248103
Fold 3 IBS: 0.17557459979421097
Fold 4 IBS: 0.1746415770408096
Fold 5 IBS: 0.1738499618927359
[I 2024

Fold 2 IBS: 0.13957734581072945
Fold 3 IBS: 0.1860801071022301
Fold 4 IBS: 0.16617625838199115
Fold 5 IBS: 0.17986715594608155
[I 2024-04-15 17:42:36,268] Trial 38 finished with value: 0.1834040501470985 and parameters: {'subsample': 0.8106303770923914, 'dropout_rate': 0.3720991365773877, 'n_estimators': 389, 'learning_rate': 0.05967708581399747}. Best is trial 35 with value: 0.17623804924912195.
Fold 1 IBS: 0.2153213199042684
Fold 2 IBS: 0.1557220858168984
Fold 3 IBS: 0.17296664592719105
Fold 4 IBS: 0.1733821578831075
Fold 5 IBS: 0.17309761052642533
[I 2024-04-15 17:42:37,128] Trial 39 finished with value: 0.17809796401157812 and parameters: {'subsample': 0.8935775772493658, 'dropout_rate': 0.40729159514619007, 'n_estimators': 144, 'learning_rate': 0.06892741183938003}. Best is trial 35 with value: 0.17623804924912195.
Fold 1 IBS: 0.2145395153982697
Fold 2 IBS: 0.1565353784804454
Fold 3 IBS: 0.17297575411525098
Fold 4 IBS: 0.17367276804259238
Fold 5 IBS: 0.17326366824345357
[I 2024-04

Fold 2 IBS: 0.14040273178208976
Fold 3 IBS: 0.17509128263968832
Fold 4 IBS: 0.16499123304160157
Fold 5 IBS: 0.17454013496230905
[I 2024-04-15 17:43:24,880] Trial 57 finished with value: 0.17795256932181863 and parameters: {'subsample': 0.8862861902385478, 'dropout_rate': 0.4359332997838259, 'n_estimators': 268, 'learning_rate': 0.0656783260329411}. Best is trial 35 with value: 0.17623804924912195.
Fold 1 IBS: 0.22698120271099279
Fold 2 IBS: 0.14410789602835403
Fold 3 IBS: 0.17207018536840524
Fold 4 IBS: 0.16639065609011652
Fold 5 IBS: 0.1721004094342305
[I 2024-04-15 17:43:26,416] Trial 58 finished with value: 0.1763300699264198 and parameters: {'subsample': 0.9333451032938084, 'dropout_rate': 0.5176718588682189, 'n_estimators': 233, 'learning_rate': 0.06122167786858218}. Best is trial 35 with value: 0.17623804924912195.
Fold 1 IBS: 0.22318872923760744
Fold 2 IBS: 0.14732818664429548
Fold 3 IBS: 0.17159998040780355
Fold 4 IBS: 0.16798209411560538
Fold 5 IBS: 0.17175813348418825
[I 2024

Fold 2 IBS: 0.16954053173030745
Fold 3 IBS: 0.1774760435677439
Fold 4 IBS: 0.18259907726741195
Fold 5 IBS: 0.17869941809076093
[I 2024-04-15 17:43:57,161] Trial 76 finished with value: 0.18327674618530906 and parameters: {'subsample': 0.9788165536606092, 'dropout_rate': 0.45549606805946546, 'n_estimators': 190, 'learning_rate': 0.03684462007229316}. Best is trial 35 with value: 0.17623804924912195.
Fold 1 IBS: 0.23442318874698415
Fold 2 IBS: 0.1404707945240763
Fold 3 IBS: 0.17496721384379338
Fold 4 IBS: 0.16506377542785522
Fold 5 IBS: 0.17436183465785568
[I 2024-04-15 17:43:59,290] Trial 77 finished with value: 0.17785736144011294 and parameters: {'subsample': 0.8943155387464099, 'dropout_rate': 0.6676023717320858, 'n_estimators': 256, 'learning_rate': 0.0683573703407812}. Best is trial 35 with value: 0.17623804924912195.
Fold 1 IBS: 0.21268481271568046
Fold 2 IBS: 0.15819098367709067
Fold 3 IBS: 0.17561877424846498
Fold 4 IBS: 0.17606931646844712
Fold 5 IBS: 0.17422981263426884
[I 202

Fold 2 IBS: 0.13948590962604285
Fold 3 IBS: 0.18096535951774095
Fold 4 IBS: 0.16548661220266322
Fold 5 IBS: 0.17800070355702288
[I 2024-04-15 17:44:40,974] Trial 95 finished with value: 0.1811539575870724 and parameters: {'subsample': 0.9762033205972742, 'dropout_rate': 0.49501037899026445, 'n_estimators': 353, 'learning_rate': 0.0607083567203958}. Best is trial 93 with value: 0.17623404026337877.
Fold 1 IBS: 0.2207576570502906
Fold 2 IBS: 0.1492854267724223
Fold 3 IBS: 0.1716590418669343
Fold 4 IBS: 0.16914562033169253
Fold 5 IBS: 0.17183582183224355
[I 2024-04-15 17:44:43,309] Trial 96 finished with value: 0.17653671357071665 and parameters: {'subsample': 0.9534755793524864, 'dropout_rate': 0.5589265767344522, 'n_estimators': 273, 'learning_rate': 0.043402912975557775}. Best is trial 93 with value: 0.17623404026337877.
Fold 1 IBS: 0.2334916712205732
Fold 2 IBS: 0.14074869741869392
Fold 3 IBS: 0.17467276277616783
Fold 4 IBS: 0.16505173801398437
Fold 5 IBS: 0.1739403109193091
[I 2024-0

In [66]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [67]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.682
train_ibs:  0.176


#### Test

In [68]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [69]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO,y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.7165502912345604,
                                              learning_rate=0.09279671898160717,
                                              n_estimators=496,
                                              random_state=123,
                                              subsample=0.985919291330434)

C-index score: 0.58


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.5332726920984945,
                                              learning_rate=0.052564680229078654,
                                              n_estimators=244,
                                              random_state=123,
                                              subsample=0.99766016156918)

IBS: 0.229


In [70]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [71]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.846,1.0
ExtraSurvivalTrees,0.791,2.0
GradientBoosting,0.765,3.0
CoxElastic,0.738,4.0
CoxPH,0.736,5.5
CoxLasso,0.736,5.5
CoxRidge,0.722,7.0
ComponentwiseGradientBoosting,0.682,8.0


In [72]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
CoxLasso,0.171,2.0
CoxElastic,0.171,2.0
Randomsurvivalforest,0.171,2.0
CoxPH,0.172,4.0
ExtraSurvivalTrees,0.176,5.5
ComponentwiseGradientBoosting,0.176,5.5
GradientBoosting,0.207,7.0
CoxRidge,0.217,8.0


In [73]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
Randomsurvivalforest,0.602,1.0
ComponentwiseGradientBoosting,0.580,2.0
GradientBoosting,0.579,3.0
ExtraSurvivalTrees,0.572,4.0
CoxPH,0.567,5.5
CoxElastic,0.567,5.5
CoxLasso,0.566,7.0
CoxRidge,0.565,8.0


In [74]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
GradientBoosting,0.218,1.0
CoxRidge,0.221,2.5
ExtraSurvivalTrees,0.221,2.5
ComponentwiseGradientBoosting,0.229,4.0
Randomsurvivalforest,0.233,5.0
CoxLasso,0.257,6.5
CoxElastic,0.257,6.5
CoxPH,0.259,8.0


In [75]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d2/os/minmax/rent/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d2_os_minmax_rent_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [76]:
from datetime import date

current_date = date.today()
print(current_date)

2024-04-15
